# Environment Setup

## Install libraries

In [ ]:
pip install transformers datasets evaluate

## Import and verify libraries

In [42]:
import torch
import transformers

print("Torch version:", torch.__version__)
print("Transformers version: ",transformers.__version__)

Torch version: 2.11.0+cu128
Transformers version:  5.10.1


## Verify GPU Availability

In [43]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


# Import Dependencies

## Import libraries

In [44]:
# import torch
from transformers import (BertTokenizer,
                          BertForSequenceClassification,
                          Trainer,
                          TrainingArguments,
                          logging)
import numpy as np
import pandas as pd
from datasets import load_dataset
import evaluate

## Fix model name

In [45]:
MODEL_NAME = "bert-base-uncased"

# Load and Inspect Dataset

## Load dataset

In [46]:
df= load_dataset("Reyansh4/Fake-News-Classification")

In [47]:
df=df['train'].to_pandas()

Convert to Pandas dataframe

## First 5 rows

In [48]:
df.shape

(20800, 5)

In [ ]:
df.head()

Text column: text <br>
Label column: label

## Datatype and other details

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20800 non-null  int64 
 1   title   20242 non-null  object
 2   author  18843 non-null  object
 3   text    20761 non-null  object
 4   label   20800 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 812.6+ KB


## Missing values

In [51]:
df["label"].isnull().sum()

np.int64(0)

No missing label values

In [52]:
df["text"].isnull().sum()

np.int64(39)

Missing text values present

## Check class imbalance

In [53]:
df["label"].value_counts(normalize=True)

,proportion
label,
1,0.500625
0,0.499375


Almost equal classes

# Data Preprocessing

## Handling missing text values

In [54]:
df.dropna(subset=["text"],inplace=True)
df["text"].isnull().sum()

np.int64(0)

In [55]:
df.shape

(20761, 5)

In [56]:
df["label"].value_counts(normalize=True)

,proportion
label,
0,0.500313
1,0.499687


Classes still almost equally balanced

## Remove unnecessary columns

In [59]:
df.drop(["id","title","author"],inplace=True, axis=1)

In [60]:
df.head()

,text,label
0,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,Ever get the feeling your life circles the rou...,0
2,"Why the Truth Might Get You Fired October 29, ...",1
3,Videos 15 Civilians Killed In Single US Airstr...,1
4,Print \nAn Iranian woman has been sentenced to...,1


# Train-Test Split

In [64]:
from sklearn.model_selection import train_test_split

In [66]:
train_df, validation_df=train_test_split(df,test_size=0.3,stratify=df["label"])

In [67]:
train_df.head()

,text,label
9633,Twitter refuses to verify the official account...,0
7320,More Reports Of Votes Flipping From Trump To C...,1
13486,The Islamic State has formally taken responsib...,0
15498,Chart Of The Day: Automobile Demographics---St...,1
13057,Ireland became the first country in the world ...,0


In [68]:
train_df.shape

(14532, 2)

In [69]:
validation_df.head()

,text,label
14234,"Elton John and his longtime boyfriend, David F...",0
5935,Donald Trump and called out Hillary Clinton fo...,1
6603,"Thursday on MSNBC’s “Morning Joe,” Sen. Rand P...",0
666,Ve la película de su vida y descubre que ha ll...,1
14265,Hillary Clinton’s bout of pneumonia and the cr...,0


In [70]:
validation_df.shape

(6229, 2)

# Tokenization

In [71]:
tokenizer= BertTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [72]:
train_encoding= tokenizer(list(train_df["text"]),padding=True, truncation=True,max_length=256,return_tensors="pt")

In [73]:
validation_encoding= tokenizer(list(validation_df["text"]),padding=True, truncation=True,max_length=256,return_tensors="pt")

In [74]:
train_encoding

{'input_ids': tensor([[  101, 10474, 10220,  ...,  5198,  2113,   102],
        [  101,  2062,  4311,  ...,  7552,  6568,   102],
        [  101,  1996,  5499,  ...,  1012,  4177,   102],
        ...,
        [  101,  2002,  2170,  ...,  1010,  1521,   102],
        [  101,  2017,  2024,  ...,  3342,  1037,   102],
        [  101,  6866,  2006,  ...,  1037,  2155,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}

In [75]:
validation_encoding

{'input_ids': tensor([[  101, 19127,  2198,  ...,  1012, 16824,   102],
        [  101,  6221,  8398,  ...,     0,     0,     0],
        [  101,  9432,  2006,  ...,  2149,  2000,   102],
        ...,
        [  101, 13503,  1517,  ...,  1037, 25381,   102],
        [  101,  3190,  1517,  ...,  2219,  1996,   102],
        [  101,  2062,  4841,  ...,  4676,  1999,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}

# Dataset Formatting for PyTorch

In [76]:
train_labels = torch.tensor(train_df["label"].values)
validation_labels = torch.tensor(validation_df["label"].values)

In [79]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [80]:
train_dataset = SimpleDataset(train_encoding, train_labels)
validation_dataset = SimpleDataset(validation_encoding, validation_labels)

In [87]:
sample = validation_dataset[0]

In [88]:
print(sample.keys())
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
torch.Size([256])
torch.Size([256])
tensor(0)


# Load Pretrained BERT Model

In [93]:
num_labels = len(set(train_df["label"]))
print("Number of labels:", num_labels)

Number of labels: 2


In [94]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [95]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)

Using device: cuda


# Training Configuration

In [99]:
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    report_to="none"
)

In [100]:
training_args.warmup_steps = 100

# Model Training

In [101]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

In [102]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")
    precision = precision_metric.compute(predictions=preds, references=labels, average="binary")
    recall = recall_metric.compute(predictions=preds, references=labels, average="binary")

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"]
    }

In [104]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics
)

In [105]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.035737,0.033975,0.991331,0.991371,0.986328,0.996466
2,0.000118,0.030727,0.994542,0.994536,0.995175,0.993897
3,0.000062,0.028694,0.995665,0.995665,0.995186,0.996145


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=5451, training_loss=0.042106017595420576, metrics={'train_runtime': 2607.1752, 'train_samples_per_second': 16.722, 'train_steps_per_second': 2.091, 'total_flos': 5735294784737280.0, 'train_loss': 0.042106017595420576, 'epoch': 3.0})

# Evaluation

In [106]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.000062,0.028694,3,0.995665,0.995665,0.995186,0.996145


{'eval_loss': 0.028694303706288338,
 'eval_accuracy': 0.9956654358645047,
 'eval_f1': 0.9956654358645047,
 'eval_precision': 0.995186136071887,
 'eval_recall': 0.9961451975586251}

# Save Model & Tokenizer

In [109]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [108]:
tokenizer.save_pretrained("./results")

('./results/tokenizer_config.json', './results/tokenizer.json')